In [ ]:
'''
Note: for all datasets and the corresponding diagnostics for each comparison, we will compute
Spearman's rank correlation to get a sense of fit within each dataset, and then report ranges across datasets.
'''
import numpy as np
import pandas as pd
import torch, os, pickle, csv
from torch.nn.functional import logsigmoid
from tqdm.autonotebook import tqdm
import matplotlib.pyplot as plt
import torch

# what is our list of datasets?
datasets = [
    "acs", "alphafold", "bodmas", "hiv", 
    "llm_evals/mmlu-pro", "llm_evals/bbh+gpqa+ifeval+math+musr", "pew", "ulb"]

# directory
if "diagnostics" not in os.listdir():
    os.mkdir("diagnostics")
    
# slightly different processing routines per type of dataset (handle "alphafold" separately)
standard_datasets = ["acs", "bodmas", "hiv", "pew", "ulb"]
llm_evals_datasets = ["llm_evals/mmlu-pro", "llm_evals/bbh+gpqa+ifeval+math+musr"]

# for CUDA
device = "cuda" if torch.cuda.is_available() else "cpu"

# 1. Diagnostics for PAI vs. SAI on all datasets.

In [ ]:
'''
Are the ML predictor's errors balanced between over- and under-prediction or predominant in one direction?
I.e., systematic. directionality of prediction error. Of course, don't trust the 1/(1-b^2) final form.
'''
# create a dataframe to store our results
b2s = pd.DataFrame(columns=["dataset", "b2"])

# b_0^2 = (\sum_i (y_i - fhat(x_i)))^2 / (\sum_i |y_i - fhat(x_i)|)^2: i.e., biased imputation
for dataset in tqdm(standard_datasets):
    
    # load in the data
    X = torch.load(f"datasets/{dataset}/X_test.pt") # (N, D)
    y = torch.load(f"datasets/{dataset}/y_test.pt") # (N,)
    beta_init = torch.load(f"datasets/{dataset}/beta_init.pt") # (D,)
    
    # compute our true residuals
    e = y - torch.sigmoid(X @ beta_init)
    
    # compute our signed-residual diagnostic + add it to the list
    b2 = (e.sum() / e.abs().sum()) ** 2
    b2s.loc[len(b2s.index)] = [dataset, b2.item()]
    
# handle alphafold separately
y = torch.load(f"datasets/alphafold/y_test.pt")
fxs = torch.load(f"datasets/alphafold/fhat_test.pt")
e = y - fxs
b2 = (e.sum() / e.abs().sum()) ** 2
b2s.loc[len(b2s.index)] = ["alphafold", b2.item()]

# handle the llm evaluation suites
for dataset in tqdm(llm_evals_datasets):
    
    # load in our data and our initial coefficients
    Y = torch.load(f"datasets/{dataset}/Y_test.pt", map_location=device) # (N_NEW, N_Q)
    X = torch.load(f"datasets/{dataset}/V.pt", map_location=device) # (N_Q, D)
    U = torch.load(f"datasets/{dataset}/U.pt", map_location=device)
    beta_init = U.mean(dim=0) # (D,)
    
    # compute residuals
    e = Y - torch.sigmoid(X @ beta_init)
    b2 = ((e.sum(axis=1) / e.abs().sum(axis=1)) ** 2).mean()
    
    # add to our list
    b2s.loc[len(b2s.index)] = [dataset, b2.item()]
    
# save our results
b2s.to_csv("diagnostics/b2s.csv", index=False)

# 2. Diagnostics for PAI vs. Uniform on all datasets.

In [ ]:
# fixed budgets
BUDGET_PROPS = torch.tensor(np.round(np.linspace(0.0, 0.25, 11)[1:], decimals=3), device=device)

# load in alphafold individually
y_test = torch.load("datasets/alphafold/y_test.pt"); N = y_test.shape[0]
fxs = torch.load("datasets/alphafold/fhat_test.pt")
theta = y_test.mean(dtype=float)
pbar = fxs.mean()

# what is our initial-step sampling policy?
qs = ((torch.minimum(fxs, 1.0 - fxs) / torch.minimum(fxs, 1.0 - fxs).sum()) * 0.5) + (0.5/N)

# get our actual budgets at this level
ns = BUDGET_PROPS * N

# variances of uniform (after multipling by n)
numerators = ((1.0 - ns / N) * theta)
denominators = (qs * ((pbar + (y_test - fxs) / (N * qs) - theta) ** 2)).sum()

# diagnostics
R = numerators / denominators

# create our dataframe
unif_diagnostics = pd.DataFrame(
    [["alphafold", BUDGET_PROPS[i].item(), R[i].item()] for i in range(R.shape[0])], 
    columns=["dataset", "budget", "R"])

In [ ]:
# go thru our standard dataset
for dataset in tqdm(standard_datasets):
    
    # load in our standard datasets
    X_test = torch.load(f"datasets/{dataset}/X_test.pt")
    y_test = torch.load(f"datasets/{dataset}/y_test.pt"); N = y_test.shape[0]
    beta_init = torch.load(f"datasets/{dataset}/beta_init.pt")
    fxs = torch.sigmoid(X_test @ beta_init)
    theta = y_test.mean(dtype=float)
    pbar = fxs.mean()
    
    # what is our initial-step sampling policy?
    qs = ((torch.minimum(fxs, 1.0 - fxs) / torch.minimum(fxs, 1.0 - fxs).sum()) * 0.5) + (0.5/N)

    # get our actual budgets at this level
    ns = BUDGET_PROPS * N

    # variances of uniform (after multipling by n)
    numerators = ((1.0 - ns / N) * theta)
    denominators = (qs * ((pbar + (y_test - fxs) / (N * qs) - theta) ** 2)).sum()

    # diagnostics
    R = numerators / denominators
    
    # package it up and add
    unif_diagnostics = pd.concat(
        [unif_diagnostics, 
        pd.DataFrame(
            [[dataset, BUDGET_PROPS[i].item(), R[i].item()] for i in range(R.shape[0])], 
            columns=["dataset", "budget", "R"])]
    )

In [ ]:
# go thru our LLM evaluation datasets
for dataset in tqdm(llm_evals_datasets):
    
    # load in alphafold individually
    X_test = torch.load(f"datasets/{dataset}/V.pt", map_location=device)
    Y_test = torch.load(f"datasets/{dataset}/Y_test.pt", map_location=device)
    beta_init = torch.load(f"datasets/{dataset}/U.pt", map_location=device).mean(axis=0)
    N = Y_test.shape[1]
    fxs = torch.sigmoid(X_test @ beta_init)
    thetas = Y_test.mean(axis=1)
    pbar = fxs.mean()
    
    # what is our initial-step sampling policy?
    qs = ((torch.minimum(fxs, 1.0 - fxs) / torch.minimum(fxs, 1.0 - fxs).sum()) * 0.5) + (0.5/N)
    
    # get our actual budgets at this level
    ns = BUDGET_PROPS * N
    
    # variances of uniform (after multipling by n)
    numerators = (1.0 - ns[None,:] / N) * thetas[:,None] # (N_NEW, N_BUDGETS)
    denominators = (qs * ((pbar + (Y_test - fxs) / (N * qs) - thetas[:,None]) ** 2)).sum(axis=1)
    
    # diagnostics (mean over all the models)
    R = (numerators / denominators[:,None]).mean(axis=0)
    
    # package it up and add
    unif_diagnostics = pd.concat(
        [unif_diagnostics, 
        pd.DataFrame(
            [[dataset, BUDGET_PROPS[i].item(), R[i].item()] for i in range(R.shape[0])], 
            columns=["dataset", "budget", "R"])]
    )
    
# save our results
unif_diagnostics.to_csv("diagnostics/unif_diagnostics.csv", index=False)

# 3. Diagnostics for PAI vs. PPAT/PPI on all datasets.

In [ ]:
# dataframe to store all of our results
fname = "ppat_diagnostics_non_llm_evals.csv"
columns = ["dataset", "pre_ybar_hat", "actual_prop_ones", "pre_acc", "pre_brier", 
           "pre_brier_mean_im1", "post_acc_mean", "post_brier_mean_im1", "post_mean_brier_im1_improvement"]
if fname not in os.listdir("diagnostics"):
    with open(f"diagnostics/{fname}", "wt") as file:
        file.write(",".join(columns) + "\n")

# diagnostic: will the first ML update help or damage the predictor? 
for dataset in tqdm(standard_datasets):
    
    # 0. can skip ULB because both PAI and PPAT failed
    if dataset == "ulb":
      continue
    
    # a. load in X_test and y_test + initial weights for OLR
    X = torch.load(f"datasets/{dataset}/X_test.pt").to(device) # (N, D)
    y = torch.load(f"datasets/{dataset}/y_test.pt").to(device) # (N,)
    beta_init = torch.load(f"datasets/{dataset}/beta_init.pt").to(device) # (D,)
    N = X.shape[0]
    
    # b. pre-update accuracy and brier
    pre_logit = X @ beta_init
    pre_acc = (((pre_logit) > 0) == y).mean(dtype=float)
    pre_yhat = torch.sigmoid(pre_logit)
    pre_brier = ((y - pre_yhat) ** 2).mean()
    pre_briers_im1 = (torch.tile(pre_yhat - y, (N,1)).fill_diagonal_(0.0) ** 2).sum(axis=1) / (N-1) # (N,)
    
    # c. compute pre-update predicted vs. actual proportions of 1s
    pre_ybar_hat, actual_prop_ones = pre_yhat.mean(dtype=float), y.mean()
    
    # d. probabilities of sampling each point at t=0 (computed stably): min(p_i, 1-p_i) + unif mix.
    # min(\sig(z), 1 - \sig(z)) = min(\sig(z), \sig(-z)) = \sig(min(z, -z)) = \sig(-|z|)
    log_u_ts = logsigmoid(-pre_logit.abs()) # (N,)
    h_js = torch.softmax(log_u_ts, dim=-1) # (N,)
    q_js = (0.5 / N) + 0.5 * h_js # (N,)
    
    # e. what happens to beta if we sample each given point? 
    w = pre_yhat[:,None] * (1.0 - pre_yhat[:,None]) # (N, 1)
    diag_precision = 1.0 + (w * (X ** 2)) # (N, D)
    betas_next = beta_init + (((y - pre_yhat)[:,None] * X) / diag_precision) # (N, D)
    
    # f. compute new accuracies and brier scores
    post_logit = betas_next @ X.T # (N, N)
    post_correct = ((post_logit > 0) == y[None,:]).float()
    post_accs = (post_correct.sum(axis=1) - post_correct.diagonal()) / (N-1) # (N,)
    post_briers = ((torch.sigmoid(post_logit) - y[None,:]).fill_diagonal_(0.0) ** 2).sum(axis=1) / (N-1)
    post_acc_mean, post_brier_mean_im1 = (post_accs * q_js).sum(), (post_briers * q_js).sum()
    
    # g. mean post-update relative change in brier score (negative means better!)
    pre_brier_mean_im1 = (pre_briers_im1 * q_js).sum()
    post_mean_brier_im1_improvement = (post_brier_mean_im1 / pre_brier_mean_im1) - 1 # negative means IMPROVE!
    
    # h. packaging: ybar_hat, ybar ...
    results = [
        dataset, pre_ybar_hat.item(), actual_prop_ones.item(), 
        pre_acc.item(), pre_brier.item(), pre_brier_mean_im1.item(), 
        post_acc_mean.item(), post_brier_mean_im1.item(), post_mean_brier_im1_improvement.item()]
    with open(f"diagnostics/{fname}", "a", newline="") as file:
        writer = csv.writer(file)
        writer.writerow(results)

In [ ]:
# dataframe to store all of our results (we'll average over models later)
fname = "ppat_diagnostics_llm_evals.csv"
columns = ["dataset", "model_idx", "pre_ybar_hat", "actual_prop_ones", "pre_acc", "pre_brier", 
           "pre_brier_mean_im1", "post_acc_mean", "post_brier_mean_im1"]
if fname not in os.listdir("diagnostics"):
    with open(f"diagnostics/{fname}", "wt") as file:
        file.write(",".join(columns) + "\n")

# repeating for the LLM datasets -- we will need to be smart with GPU memory, for-loop over models
for dataset in tqdm(llm_evals_datasets):
    
    # load in initial factor model fits for LLM evaluations
    U = torch.load(f"datasets/{dataset}/U.pt", map_location=device).to(device) # (N_OLD, D)
    X = torch.load(f"datasets/{dataset}/V.pt", map_location=device).to(device) # (N, D)
    ys = torch.load(f"datasets/{dataset}/Y_test.pt", map_location=device).to(device) # (N_NEW, N)
    
    # initialize new model factor mean & SD with empirical counterparts
    beta_init = U.mean(axis=0) # (D,)
    diag_precisions = 1.0 / (U.std(axis=0) ** 2) # (D,)
    
    # get our finite-population size + no. of new models
    N_NEW, N = ys.shape; D = U.shape[1]
    
    # iterate thru each of our 2.2K models
    for m_idx in tqdm(range(N_NEW), desc="Model"):
        
        # a. get our y for this model
        y = ys[m_idx] # (N,)
        
        # b. pre-update accuracy and brier
        pre_logit = X @ beta_init
        pre_acc = (((pre_logit) > 0) == y).mean(dtype=float)
        pre_yhat = torch.sigmoid(pre_logit)
        pre_brier = ((y - pre_yhat) ** 2).mean()
        pre_briers_im1 = (torch.tile(pre_yhat - y, (N,1)).fill_diagonal_(0.0) ** 2).sum(axis=1) / (N-1) # (N,)

        # c. compute pre-update predicted vs. actual proportions of 1s
        pre_ybar_hat, actual_prop_ones = pre_yhat.mean(dtype=float), y.mean()

        # d. probabilities of sampling each point at t=0 (computed stably): min(p_i, 1-p_i) + unif mix.
        # min(\sig(z), 1 - \sig(z)) = min(\sig(z), \sig(-z)) = \sig(min(z, -z)) = \sig(-|z|)
        log_u_ts = logsigmoid(-pre_logit.abs()) # (N,)
        h_js = torch.softmax(log_u_ts, dim=-1) # (N,)
        q_js = (0.5 / N) + 0.5 * h_js # (N,)

        # e. what happens to beta if we sample each given point? 
        w = pre_yhat[:,None] * (1.0 - pre_yhat[:,None]) # (N, 1)
        diag_precision_new = diag_precisions + (w * (X ** 2)) # (N, D)
        betas_next = beta_init + (((y - pre_yhat)[:,None] * X) / diag_precision_new) # (N, D)

        # f. compute new accuracies and brier scores
        post_logit = betas_next @ X.T # (N, N)
        post_correct = ((post_logit > 0) == y[None,:]).float()
        post_accs = (post_correct.sum(axis=1) - post_correct.diagonal()) / (N-1) # (N,)
        post_briers = ((torch.sigmoid(post_logit) - y[None,:]).fill_diagonal_(0.0) ** 2).sum(axis=1) / (N-1)
        post_acc_mean, post_brier_mean_im1 = (post_accs * q_js).sum(), (post_briers * q_js).sum()

        # g. mean post-update relative change in brier score needs pre-change reference
        pre_brier_mean_im1 = (pre_briers_im1 * q_js).sum()

        # h. packaging: ybar_hat, ybar ...
        results = [
            dataset, m_idx, pre_ybar_hat.item(), actual_prop_ones.item(), 
            pre_acc.item(), pre_brier.item(), pre_brier_mean_im1.item(), 
            post_acc_mean.item(), post_brier_mean_im1.item()]
        with open(f"diagnostics/{fname}", "a", newline="") as file:
            writer = csv.writer(file)
            writer.writerow(results)

In [ ]:
# stitching together the two types of datasets
df1 = pd.read_csv("diagnostics/ppat_diagnostics_non_llm_evals.csv")
df2 = pd.read_csv("diagnostics/ppat_diagnostics_llm_evals.csv")

# average over LLM new test models
df2 = df2.groupby(["dataset"]).mean().reset_index().drop(columns=["model_idx"])
df2["post_mean_brier_im1_improvement"] = (df2.post_brier_mean_im1 / df2.pre_brier_mean_im1) - 1.0

# save our results
pd.concat([df1, df2]).to_csv("diagnostics/ppat_diagnostics_combined.csv", index=False)

# 4. Diagnostic for coverage on ULB only.

In [ ]:
# test X's, true proportion of y=1, OLR coefficients
X_test = torch.load("datasets/ulb/X_test.pt")
y_test = torch.load("datasets/ulb/y_test.pt")
beta_init = torch.load("datasets/ulb/beta_init.pt")
yhat_test = torch.sigmoid(X_test @ beta_init)

# what is the true proportion of y=1? total of 223 frauds in test set out of 142404 datapoints.
print(f"1. True proportion of y's in test set: {y_test.mean()}.")
print(f"2. Predicted proportion of y's in test set: {(yhat_test > 0.5).mean(dtype=float)}.")
print(f"3. Number of y=1 datapoints where ML model predicted <1% chance of fraud: {(yhat_test[y_test == 1] < 0.01).sum()}.")
print(f"4. Number of y=1 datapoints where ML model predicted <0.1% chance of fraud: {(yhat_test[y_test == 1] < 0.001).sum()}.")